## Requirements

This section lists the system requirements and Python libraries required to run this notebook.

### Python Libraries (from VibeASR.cpp)
* `huggingface_hub` (Model download)
* `gradio` (Demo GUI)
* `soundfile` (Audio IO)
* `numpy` (Numerical operations)
* `llama.cpp` conversion dependencies (including `sentencepiece` and `gguf` converter tools)

In [ ]:
# Force-upgrade numpy to version 2+ first to satisfy PyTorch/Transformers/Kokoro ABI compilation standards
!pip install --upgrade "numpy>=2.0.0"
!pip install pyngrok fastapi uvicorn pydantic soundfile pydub kokoro sounddevice

In [ ]:
import os
# Programmatically exit the current process. Google Colab will automatically restart the runtime kernel.
os._exit(0)

In [ ]:
import os

requirements_path = '/content/VibeASR.cpp/requirements.txt'
if os.path.exists(requirements_path):
    with open(requirements_path, 'r') as f:
        requirements = f.read().splitlines()
    print("Requirements found in VibeASR.cpp:")
    for req in requirements:
        print(f"- {req}")
else:
    print("No requirements.txt found in the repository.")

In [ ]:
import io
import os
import shutil
import threading
import subprocess
from pathlib import Path
from contextlib import asynccontextmanager

# Ensure clean, upgraded imports
import numpy as np
import soundfile as sf
from pydub import AudioSegment
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse
from kokoro import KPipeline

# --- 1. Global Configurations ---
VIBE_ASR_BIN = "./build/bin/asr_infer"
VAE_MODEL_PATH = "models/vibeasr/vibeasr-vae-encoder-i8_s.gguf"
LM_MODEL_PATH = "models/vibeasr/vibeasr-lm-i2_s-embed-q6_k.gguf"
UPLOAD_DIR = Path("/tmp/speech_forwarding")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# State container for lazy global loading
state = {"pipeline": None}

# --- 2. FastAPI Lifespan (Ensures Stable Pipeline Boot) ---
@asynccontextmanager
async def lifespan(app: FastAPI):
    print("⏳ Initializing system dependencies and model pipelines...")

    # Check for C++ runtime binary presence
    if not os.path.exists(VIBE_ASR_BIN):
        print(f"❌ CRITICAL WARNING: VibeASR binary not found at {VIBE_ASR_BIN}. Build it before sending requests.")

    try:
        # Pre-initialize Kokoro to avoid cold-start delays on incoming API calls
        state["pipeline"] = KPipeline(lang_code='a')
        print("✅ Kokoro TTS Engine compiled and ready.")
    except Exception as e:
        print(f"❌ CRITICAL ERROR initializing Kokoro TTS: {str(e)}")
        print("💡 Hint: Ensure 'espeak-ng' is installed via: !apt-get install -y espeak-ng")

    yield
    print("🧹 Shutting down speech engine...")
    state["pipeline"] = None

app = FastAPI(
    title="Colab Speech-to-Speech Forwarding Backend",
    lifespan=lifespan
)

def normalize_audio(input_path: Path, output_path: Path):
    """Downsamples incoming audio to 16kHz mono layout expected by VibeASR."""
    audio = AudioSegment.from_file(str(input_path))
    audio = audio.set_frame_rate(16000).set_channels(1)
    audio.export(str(output_path), format="wav")

# --- 3. Clean API Routes ---
@app.get("/")
async def root():
    return {"status": "online", "engine": "VibeASR.cpp + Kokoro TTS"}

@app.post("/forward-speech")
async def forward_speech(file: UploadFile = File(...), voice: str = "af_sarah"):
    if not state["pipeline"]:
        raise HTTPException(status_code=500, detail="TTS Pipeline backend is uninitialized or crashed.")

    temp_input_path = UPLOAD_DIR / file.filename
    temp_normalized_path = UPLOAD_DIR / f"normalized_{file.filename}.wav"

    # Stream incoming buffer to scratch disk safely
    with temp_input_path.open("wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    try:
        normalize_audio(temp_input_path, temp_normalized_path)

        # Build optimized parallel execution runtime matrix
        command = [
            VIBE_ASR_BIN,
            "--vae-model", VAE_MODEL_PATH,
            "--lm-model", LM_MODEL_PATH,
            "--audio", str(temp_normalized_path),
            "-t", "2"  # Allocates safe execution boundaries inside shared Colab CPUs
        ]

        # Subprocess call to the ultra-fast C++ binary runtime engine
        process = subprocess.run(command, capture_output=True, text=True, check=True)
        transcribed_text = process.stdout.strip()

        print(f"📝 Transcribed text stream: '{transcribed_text}'")

        if not transcribed_text:
            raise HTTPException(status_code=422, detail="ASR Engine generated empty text matrix.")

        # Pass plaintext straight to Kokoro Generator
        generator = state["pipeline"](transcribed_text, voice=voice, speed=1.0)
        audio_chunks = [audio_chunk for _, _, audio_chunk in generator]
        full_audio_array = np.concatenate(audio_chunks)

        # Build clean 24kHz stream output buffer in memory
        memory_buffer = io.BytesIO()
        sf.write(memory_buffer, full_audio_array, 24000, format='WAV')
        memory_buffer.seek(0)

        return StreamingResponse(
            memory_buffer,
            media_type="audio/wav",
            headers={
                "X-Transcribed-Text": transcribed_text,
                "Access-Control-Expose-Headers": "X-Transcribed-Text"
            }
        )

    except subprocess.CalledProcessError as sub_err:
        print(f"❌ Subprocess Crash: {sub_err.stderr}")
        raise HTTPException(status_code=500, detail=f"VibeASR Executable Error: {sub_err.stderr}")
    except Exception as e:
        print(f"❌ General Exception: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        # Scrub temporary IO storage assets
        if temp_input_path.exists(): temp_input_path.unlink()
        if temp_normalized_path.exists(): temp_normalized_path.unlink()

# --- 4. Robust Server & Tunnel Lifecycle Management ---
def launch_pipeline(token: str):
    # Wipe any stale or orphaned proxy tunnels
    try:
        ngrok.kill()
    except Exception:
        pass

    ngrok.set_auth_token(token)

    # Establish public tunnel pointing to internal network loopback
    public_url = ngrok.connect(8000)
    print(f"\n🚀 API LIVE! Forward requests directly to your custom subdomain target:")
    print(f"➡️  {public_url}  ◄──\n")

    def run_uvicorn():
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

    # Spin execution framework in background worker context
    srv_thread = threading.Thread(target=run_uvicorn, daemon=True)
    srv_thread.start()

# --- 5. Trigger Execution ---
# Replace with your actual free token string from ://ngrok.com
NGROK_AUTHTOKEN = "2vHUbZqM42aMc8mzQLnAutH0qDy_2oWMgbfXRqCYFhdpR1ELN"
launch_pipeline(NGROK_AUTHTOKEN)

### 🚀 FastAPI + Kokoro + VibeASR Speech-to-Speech Server is Running!
You can use the public URL displayed in the server terminal log above to route your API speech forwarding requests.